In [1]:
import pandas as pd
import json

import os
from text2cypher.evaluation.runtime import jaccard
from text2cypher.evaluation.psj_similarity import provenance_subgraph_jaccard_similarity

from langchain_neo4j import Neo4jGraph

from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

c:\Users\nicol\Desktop\text-to-cypher\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
predictions_path = "..\\data\\interim\\neo4j-2024v1\\gpt-4.1\\test-trajectories.csv"
dataset_path = "..\\data\\raw\\neo4j-2024v1\\test-00000-of-00001.parquet"
eval_path = "../data/interim/evaluations/gpt-4.1/test.csv"

In [3]:
dataset_df = pd.read_parquet(dataset_path)
dataset_df['df.index'] = dataset_df.index
dataset_df

,question,schema,cypher,data_source,instance_id,database_reference_alias,df.index
0,Identify the 5 suppliers with the highest aver...,Node properties:\n- **Product**\n - `productN...,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) WI...,neo4jLabs_synthetic_gpt4o,instance_id_44523,neo4jlabs_demo_db_northwind,0
1,What are the names of the technicians that hav...,"{""ASSIGNED_TO"": {""count"": 27, ""properties"": {}...",MATCH (t:Technician) WHERE NOT EXISTS ((:Repai...,neo4j_text2cypher2023_test,instance_id_3724,None,1
2,Fetch unique values of label and description f...,Graph schema: Relevant node labels and their p...,MATCH (n:Topic) WHERE NOT n.label STARTS WITH ...,neo4jLabs_functional_cypher,instance_id_19672,None,2
3,What is the total number of companies?,"{""GasStation"": {""count"": 11, ""labels"": [], ""pr...",MATCH (c:Company) RETURN count(c),neo4j_text2cypher2023_test,instance_id_3342,None,3
4,List the articles that mention the organizatio...,Node properties:\n- **Person**\n - `name`: ST...,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,neo4jLabs_synthetic_claudeopus,instance_id_37923,neo4jlabs_demo_db_companies,4
...,...,...,...,...,...,...,...
4828,Find nodes that are at the end of a path start...,Graph schema: Relevant node labels and their p...,MATCH (a:Topic{description:'The study of how s...,neo4jLabs_functional_cypher,instance_id_6886,None,4828
4829,What is the average number of followers for us...,Node properties:\n- **User**\n - `betweenness...,MATCH (neo4j:User {screen_name: 'neo4j'})<-[:F...,neo4jLabs_synthetic_gemini,instance_id_34538,neo4jlabs_demo_db_twitter,4829
4830,Retrieve the Article where comments or abstrac...,Graph schema: Relevant node labels and their p...,MATCH (n:Article) WHERE n.comments CONTAINS 'P...,neo4jLabs_functional_cypher,instance_id_18745,None,4830
4831,Find the business with the most reviews.,Node properties:\n- **Business**\n - `address...,MATCH (b:Business)<-[:REVIEWS]-(review:Review)...,neo4jLabs_synthetic_gemini,instance_id_33217,neo4jlabs_demo_db_grandstack,4831


In [4]:
predictions_df = pd.read_csv(predictions_path)
predictions_df

,df.index,json_trajectory,generated_cypher
0,0.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)\nW...
1,4.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",MATCH (a:Article)-[:MENTIONS]->(o:Organization...
2,11.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m...
3,12.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...","MATCH (u:User)-[:VIP]->(s:Stream) WITH u, coun..."
4,13.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",MATCH (t:Tweet) RETURN t.id AS tweetId ORDER B...
...,...,...,...
2466,4683.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2467,4699.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2468,4776.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2469,4807.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN


In [5]:
# show rows having generated_cypher as NaN
predictions_df[predictions_df['generated_cypher'].isna()]

,df.index,json_trajectory,generated_cypher
2202,31.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2203,80.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2204,220.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2205,323.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2206,329.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
...,...,...,...
2466,4683.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2467,4699.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2468,4776.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN
2469,4807.0,"[{""role"": ""system"", ""content"": ""## ROLE AND TA...",NaN


In [6]:
eval_df = pd.read_csv(eval_path)
eval_df

,df.index,jaccard_similarity,psj_similarity,is_executable,is_empty
0,0.0,1.000000,1.0,True,False
1,4.0,0.000000,1.0,True,False
2,11.0,0.333333,1.0,True,False
3,12.0,1.000000,1.0,True,False
4,13.0,1.000000,1.0,True,False
...,...,...,...,...,...
2466,4683.0,0.000000,0.0,False,NaN
2467,4699.0,0.000000,0.0,False,NaN
2468,4776.0,0.000000,0.0,False,NaN
2469,4807.0,0.000000,0.0,False,NaN


In [7]:
temp_df = predictions_df[['df.index', 'generated_cypher']].merge(
    dataset_df[['df.index', 'database_reference_alias', 'cypher']],
    on='df.index',
    how='left'
)
temp_df

,df.index,generated_cypher,database_reference_alias,cypher
0,0.0,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)\nW...,neo4jlabs_demo_db_northwind,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) WI...
1,4.0,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,neo4jlabs_demo_db_companies,MATCH (a:Article)-[:MENTIONS]->(o:Organization...
2,11.0,MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m...,neo4jlabs_demo_db_eoflix,"MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WITH g,..."
3,12.0,"MATCH (u:User)-[:VIP]->(s:Stream) WITH u, coun...",neo4jlabs_demo_db_twitch,MATCH (user:User)-[:VIP]->(stream:Stream) WITH...
4,13.0,MATCH (t:Tweet) RETURN t.id AS tweetId ORDER B...,neo4jlabs_demo_db_twitter,MATCH (t:Tweet) RETURN t.id ORDER BY t.created...
...,...,...,...,...
2466,4683.0,NaN,neo4jlabs_demo_db_network,MATCH (app:Application {name: 'logstash'})-[:D...
2467,4699.0,NaN,neo4jlabs_demo_db_network,MATCH (s:Software)-[:DEPENDS_ON]->(:Applicatio...
2468,4776.0,NaN,neo4jlabs_demo_db_stackoverflow2,MATCH (u:User)-[:ASKED]->(q:Question)<-[:ANSWE...
2469,4807.0,NaN,neo4jlabs_demo_db_companies,MATCH (p:Person)<-[:HAS_CEO]-(o:Organization)-...


In [8]:
temp_df = temp_df.merge(
    eval_df,
    on='df.index',
    how='left'
)
temp_df

,df.index,generated_cypher,database_reference_alias,cypher,jaccard_similarity,psj_similarity,is_executable,is_empty
0,0.0,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)\nW...,neo4jlabs_demo_db_northwind,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) WI...,1.000000,1.0,True,False
1,4.0,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,neo4jlabs_demo_db_companies,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,0.000000,1.0,True,False
2,11.0,MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m...,neo4jlabs_demo_db_eoflix,"MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WITH g,...",0.333333,1.0,True,False
3,12.0,"MATCH (u:User)-[:VIP]->(s:Stream) WITH u, coun...",neo4jlabs_demo_db_twitch,MATCH (user:User)-[:VIP]->(stream:Stream) WITH...,1.000000,1.0,True,False
4,13.0,MATCH (t:Tweet) RETURN t.id AS tweetId ORDER B...,neo4jlabs_demo_db_twitter,MATCH (t:Tweet) RETURN t.id ORDER BY t.created...,1.000000,1.0,True,False
...,...,...,...,...,...,...,...,...
2466,4683.0,NaN,neo4jlabs_demo_db_network,MATCH (app:Application {name: 'logstash'})-[:D...,0.000000,0.0,False,NaN
2467,4699.0,NaN,neo4jlabs_demo_db_network,MATCH (s:Software)-[:DEPENDS_ON]->(:Applicatio...,0.000000,0.0,False,NaN
2468,4776.0,NaN,neo4jlabs_demo_db_stackoverflow2,MATCH (u:User)-[:ASKED]->(q:Question)<-[:ANSWE...,0.000000,0.0,False,NaN
2469,4807.0,NaN,neo4jlabs_demo_db_companies,MATCH (p:Person)<-[:HAS_CEO]-(o:Organization)-...,0.000000,0.0,False,NaN


In [9]:
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2471 entries, 0 to 2470
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   df.index                  2471 non-null   float64
 1   generated_cypher          2202 non-null   object 
 2   database_reference_alias  2471 non-null   object 
 3   cypher                    2471 non-null   object 
 4   jaccard_similarity        2471 non-null   float64
 5   psj_similarity            2471 non-null   float64
 6   is_executable             2471 non-null   bool   
 7   is_empty                  2108 non-null   object 
dtypes: bool(1), float64(3), object(4)
memory usage: 137.7+ KB


In [10]:
temp_mask = temp_df['jaccard_similarity'].isna() | temp_df['psj_similarity'].isna()
temp_df[temp_mask]

,df.index,generated_cypher,database_reference_alias,cypher,jaccard_similarity,psj_similarity,is_executable,is_empty


In [11]:
from text2cypher.evaluation.runtime import check_validity

for idx, row in tqdm(temp_df.iterrows(), total=len(temp_df)):
    generated_cypher = row['generated_cypher']
    # check if generated_cypher is not NaN
    if pd.notna(generated_cypher):
        if pd.isna(row['is_executable']):
            database = row['database_reference_alias'].split('_')[-1]
            if database == 'eoflix':
                database = 'neoflix'

            neo4j = Neo4jGraph(
                url=os.getenv("NEO4J_URL"),
                username=database,
                password=database,
                database=database,
                timeout=1
            )
            
            is_executable, is_empty = check_validity(
                generated_cypher,
                neo4j_driver=neo4j._driver,
                database=database,
                timeout=1,
            )
            temp_df.at[idx, 'is_executable'] = is_executable
            temp_df.at[idx, 'is_empty'] = is_empty
            # print(f"Row {idx}: is_executable={is_executable}, is_empty={is_empty}")

            if is_executable:
                if pd.isna(row['jaccard_similarity']):
                    js = jaccard(row['cypher'], generated_cypher, neo4j)
                    temp_df.at[idx, 'jaccard_similarity'] = js
                if pd.isna(row['psj_similarity']):
                    psj = provenance_subgraph_jaccard_similarity(row['cypher'], generated_cypher, neo4j)
                    temp_df.at[idx, 'psj_similarity'] = psj
            else:
                temp_df.at[idx, 'jaccard_similarity'] = 0.0
                temp_df.at[idx, 'psj_similarity'] = 0.0
    else:
        temp_df.at[idx, 'jaccard_similarity'] = None
        temp_df.at[idx, 'psj_similarity'] = None
        temp_df.at[idx, 'is_executable'] = None
        temp_df.at[idx, 'is_empty'] = None

temp_df

  0%|          | 0/2471 [00:00<?, ?it/s]C:\Users\nicol\AppData\Local\Temp\ipykernel_6704\1943032077.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  temp_df.at[idx, 'is_executable'] = None
100%|██████████| 2471/2471 [00:00<00:00, 9955.35it/s] 


,df.index,generated_cypher,database_reference_alias,cypher,jaccard_similarity,psj_similarity,is_executable,is_empty
0,0.0,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)\nW...,neo4jlabs_demo_db_northwind,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product) WI...,1.000000,1.0,True,False
1,4.0,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,neo4jlabs_demo_db_companies,MATCH (a:Article)-[:MENTIONS]->(o:Organization...,0.000000,1.0,True,False
2,11.0,MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m...,neo4jlabs_demo_db_eoflix,"MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WITH g,...",0.333333,1.0,True,False
3,12.0,"MATCH (u:User)-[:VIP]->(s:Stream) WITH u, coun...",neo4jlabs_demo_db_twitch,MATCH (user:User)-[:VIP]->(stream:Stream) WITH...,1.000000,1.0,True,False
4,13.0,MATCH (t:Tweet) RETURN t.id AS tweetId ORDER B...,neo4jlabs_demo_db_twitter,MATCH (t:Tweet) RETURN t.id ORDER BY t.created...,1.000000,1.0,True,False
...,...,...,...,...,...,...,...,...
2466,4683.0,NaN,neo4jlabs_demo_db_network,MATCH (app:Application {name: 'logstash'})-[:D...,NaN,NaN,None,None
2467,4699.0,NaN,neo4jlabs_demo_db_network,MATCH (s:Software)-[:DEPENDS_ON]->(:Applicatio...,NaN,NaN,None,None
2468,4776.0,NaN,neo4jlabs_demo_db_stackoverflow2,MATCH (u:User)-[:ASKED]->(q:Question)<-[:ANSWE...,NaN,NaN,None,None
2469,4807.0,NaN,neo4jlabs_demo_db_companies,MATCH (p:Person)<-[:HAS_CEO]-(o:Organization)-...,NaN,NaN,None,None


In [12]:
temp_df[['df.index', 'jaccard_similarity', 'psj_similarity', 'is_executable', 'is_empty']].to_csv(
    eval_path, index=False
)